# Imports

In [3]:
import numpy as np
from tensorflow.keras.datasets import mnist

from methods.bcgd_gauss_southwell import gs_bcgd
from methods.bcgd_randomized import randomized_bcgd
from methods.gradient_descent import gradient_descent
from methods.utils.accuracy import accuracy
from methods.utils.block_size import block_step_size
from methods.utils.gradient import MultiLogisticRegressionOracle, CachedMultiLogisticRegressionOracle

rng = np.random.default_rng(seed=42) # Turnoff or modify if no seeding will be used.

I0000 00:00:1777275458.240264  516424 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777275460.884035  516424 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777275468.078955  516424 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# Data Loading

## Synthetic Data

In [4]:
n_categories = 50
n_features = 500
n_examples = 1000

syntetic_features = rng.random((n_examples, n_features))
X = rng.random((n_features, n_categories))
E = rng.random((n_examples, n_categories))

transformation = np.dot(syntetic_features,X) + E
syntetic_labels = np.argmax(transformation, axis= 1)

## MNIST Dataset

In [8]:
(mnist_features, mnist_labels), (mnist_features_test, mnist_features_test) = mnist.load_data()

mnist_features = mnist_features.reshape((mnist_features.shape[0],-1))
mnist_features_test = mnist_features_test.reshape((mnist_features_test.shape[0],-1))

In [9]:
mnist_features.shape

(60000, 784)

## Bloodcell Dataset

(60000, 784)

# Method Calls
This section is for implementing all the methods that will be used for generating graphs or any other types of solutions

In [5]:
active_features = syntetic_features
active_labels = syntetic_labels

number_of_categories = n_categories
number_of_features = n_features

#calculated step sizes
step_sizes = block_step_size(active_features)

oracle = MultiLogisticRegressionOracle(active_features, active_labels, number_of_categories)
cached_oracle = CachedMultiLogisticRegressionOracle(active_features, active_labels, number_of_categories)

## Gradient Descent

In [7]:
gradient_descent_results = gradient_descent(
    np.zeros((number_of_features, number_of_categories)),
    oracle,
    0.001,
    tol=1,
    debug=True,
)

Iteration 500: Loss = 30.417784037552337, Accuracy = 0.996
Iteration 1000: Loss = 7.501471495230362, Accuracy = 1.0
Iteration 1500: Loss = 4.993564833646815, Accuracy = 1.0
Iteration 2000: Loss = 3.8789615245186724, Accuracy = 1.0
Iteration 2500: Loss = 3.2260674296776415, Accuracy = 1.0


## BCGD Randomized

In [8]:
randomized_bcgd_results = randomized_bcgd(
    np.zeros((number_of_features, number_of_categories)),
    oracle,
    [0.011]*n_features,
    tol=0.001,
    debug=True,
)

iteration: 0, sequential distance: 2.586198938527228, loss: 3338.4298127768798
accuracy: 0.355
iteration: 500, sequential distance: 0.08493628821419394, loss: 1458.0864530009912
accuracy: 0.548
iteration: 1000, sequential distance: 0.05844303468528697, loss: 1151.0309662308691
accuracy: 0.647
iteration: 1500, sequential distance: 0.08507864076491735, loss: 940.8652237949927
accuracy: 0.739
iteration: 2000, sequential distance: 0.038920165700193396, loss: 779.8132680500894
accuracy: 0.8
iteration: 2500, sequential distance: 0.04551039504576091, loss: 661.9813986022145
accuracy: 0.856
iteration: 3000, sequential distance: 0.0409549511033515, loss: 570.1281129724466
accuracy: 0.887
iteration: 3500, sequential distance: 0.05040713925639549, loss: 494.72752811883765
accuracy: 0.911
iteration: 4000, sequential distance: 0.05073538902348513, loss: 437.4048333688297
accuracy: 0.937
iteration: 4500, sequential distance: 0.028533726505814226, loss: 388.9866522339453
accuracy: 0.956
iteration: 50

## Gauss-Southwell BCGD

In [7]:
gs_bcgd_results = gs_bcgd(
    np.zeros((number_of_features, number_of_categories)),
    cached_oracle,
    [0.011]*n_features,
    tol=0.001,
    debug=True,
)

iteration: 500, selected_block: 256, sequential distance: 0.010694185138694237, loss: 3830.806259
accuracy: 95.60%
iteration: 1000, selected_block: 42, sequential distance: 0.01021747908562482, loss: 3755.487742
accuracy: 99.60%
iteration: 1500, selected_block: 160, sequential distance: 0.009715602590937579, loss: 3681.974536
accuracy: 99.90%
iteration: 2000, selected_block: 41, sequential distance: 0.00933708434637868, loss: 3613.796183
accuracy: 99.90%
iteration: 2500, selected_block: 74, sequential distance: 0.009268431851915733, loss: 3546.985597
accuracy: 99.90%
iteration: 3000, selected_block: 107, sequential distance: 0.00859909922431902, loss: 3483.948956
accuracy: 100.00%
iteration: 3500, selected_block: 492, sequential distance: 0.008330512092741377, loss: 3424.395459
accuracy: 100.00%
iteration: 4000, selected_block: 277, sequential distance: 0.0081811480502398, loss: 3365.690692
accuracy: 100.00%
iteration: 4500, selected_block: 268, sequential distance: 0.00787060974911466